# VCF Analysis with cyvcf2

This notebook demonstrates post-variant-calling analysis:
- Variant type breakdown (SNP, indel, MNP)
- Ti/Tv ratio (quality metric for SNP calling)
- Allele frequency spectrum
- Variant filtering by INFO fields
- Genotype concordance concepts

## Ti/Tv Ratio

Transitions (Ti): purine↔purine (A↔G) or pyrimidine↔pyrimidine (C↔T)  
Transversions (Tv): purine↔pyrimidine (A/G↔C/T)

**Expected Ti/Tv ratios:**
- Random mutations: 0.5 (2 Ti vs 4 Tv possible)
- WGS across whole genome: **2.0–2.1**
- WGS in coding regions (CDS): **2.8–3.0** (selection pressure against transversions)

If your Ti/Tv is much lower than 2.0, you likely have excess false positives (transversions).
If it is much higher, check for reference bias or BQSR issues.

In [ ]:
import cyvcf2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

VCF_PATH = "../results/variants/NA12878.final.vcf.gz"

In [ ]:
# --- Collect variant statistics ---

TRANSITIONS = {('A','G'), ('G','A'), ('C','T'), ('T','C')}

records = []

try:
    vcf = cyvcf2.VCF(VCF_PATH)

    for v in vcf:
        filt = v.FILTER or 'PASS'

        # Variant type
        alt = v.ALT[0] if v.ALT else ''
        if len(v.REF) == 1 and len(alt) == 1:
            vtype = 'SNP'
            ti_tv = 'Ti' if (v.REF, alt) in TRANSITIONS else 'Tv'
        elif len(v.REF) > len(alt):
            vtype, ti_tv = 'DEL', 'NA'
        elif len(v.REF) < len(alt):
            vtype, ti_tv = 'INS', 'NA'
        else:
            vtype, ti_tv = 'MNP', 'NA'

        # FORMAT fields
        gq_arr = v.format('GQ')
        ad_arr = v.format('AD')
        dp_arr = v.format('DP')

        gq = float(gq_arr[0][0]) if gq_arr is not None else None
        dp = float(dp_arr[0][0]) if dp_arr is not None else None
        af = None
        if ad_arr is not None and ad_arr.shape[1] >= 2:
            ref_dp, alt_dp = ad_arr[0][0], ad_arr[0][1]
            total = ref_dp + alt_dp
            if total > 0:
                af = alt_dp / total

        # Genotype
        gt = v.genotypes[0][:2] if v.genotypes else (None, None)
        if gt[0] == gt[1]:
            zygosity = 'hom_alt' if gt[0] != 0 else 'hom_ref'
        else:
            zygosity = 'het'

        records.append({
            'chrom': v.CHROM, 'pos': v.POS,
            'ref': v.REF, 'alt': alt,
            'qual': v.QUAL, 'filter': filt,
            'vtype': vtype, 'ti_tv': ti_tv,
            'gq': gq, 'dp': dp, 'af': af,
            'zygosity': zygosity,
            'qd': v.INFO.get('QD'),
        })

    df = pd.DataFrame(records)
    print(f"Total variants: {len(df):,}")
    print(f"\nBy filter:")
    print(df['filter'].value_counts().to_string())

except FileNotFoundError:
    print(f"VCF not found: {VCF_PATH}. Run the pipeline first.")
    df = pd.DataFrame()

In [ ]:
if df.empty:
    print("No data to plot — run the pipeline first.")
else:
    pass_df = df[df['filter'] == 'PASS']
    print(f"PASS variants: {len(pass_df):,}")
    print(f"\nBy variant type:")
    print(pass_df['vtype'].value_counts().to_string())

    # Ti/Tv ratio
    snp_df = pass_df[pass_df['vtype'] == 'SNP']
    ti_count = (snp_df['ti_tv'] == 'Ti').sum()
    tv_count = (snp_df['ti_tv'] == 'Tv').sum()
    titv = ti_count / tv_count if tv_count > 0 else float('inf')
    print(f"\nTi/Tv ratio: {titv:.3f} (Ti={ti_count:,}, Tv={tv_count:,})")
    if 1.8 <= titv <= 2.3:
        print("  ✓ Within expected range for WGS (2.0–2.1)")
    else:
        print("  ⚠ Outside expected range — check filtering or pipeline config")

    # Zygosity
    print(f"\nZygosity (PASS SNPs):")
    print(snp_df['zygosity'].value_counts().to_string())
    het_hom = (snp_df['zygosity'] == 'het').sum() / (snp_df['zygosity'] == 'hom_alt').sum()
    print(f"Het/Hom ratio: {het_hom:.2f} (expected ~1.5–2.0 for a diploid sample)")

In [ ]:
if not df.empty:
    pass_df = df[df['filter'] == 'PASS']
    snp_df = pass_df[pass_df['vtype'] == 'SNP']

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    # Variant type breakdown
    vtype_counts = pass_df['vtype'].value_counts()
    axes[0][0].bar(vtype_counts.index, vtype_counts.values, color=['steelblue','darkorange','green','red'])
    axes[0][0].set_title('Variant types (PASS)')

    # Ti/Tv
    ti_tv_counts = snp_df['ti_tv'].value_counts()
    axes[0][1].pie(ti_tv_counts.values, labels=ti_tv_counts.index,
                   autopct='%1.1f%%', colors=['steelblue','darkorange'])
    axes[0][1].set_title(f'Ti/Tv = {titv:.2f}')

    # Zygosity
    zyg_counts = pass_df[pass_df['vtype'].isin(['SNP','DEL','INS'])]['zygosity'].value_counts()
    axes[0][2].bar(zyg_counts.index, zyg_counts.values, color='steelblue')
    axes[0][2].set_title('Zygosity (PASS variants)')

    # Allele frequency spectrum
    af_data = pass_df['af'].dropna()
    axes[1][0].hist(af_data, bins=25, color='steelblue')
    axes[1][0].axvline(x=0.5, color='red', linestyle='--', label='AF=0.5 (het)')
    axes[1][0].set_title('Allele frequency spectrum (PASS)')
    axes[1][0].set_xlabel('Alt allele frequency')
    axes[1][0].legend()

    # Depth distribution
    dp_data = pass_df['dp'].dropna()
    axes[1][1].hist(dp_data, bins=40, range=(0, dp_data.quantile(0.99)), color='steelblue')
    axes[1][1].axvline(x=dp_data.median(), color='red', linestyle='--',
                       label=f'Median={dp_data.median():.0f}x')
    axes[1][1].set_title('Read depth at PASS variant sites')
    axes[1][1].set_xlabel('Depth')
    axes[1][1].legend()

    # GQ distribution
    gq_data = pass_df['gq'].dropna()
    axes[1][2].hist(gq_data, bins=30, color='steelblue')
    axes[1][2].axvline(x=30, color='orange', linestyle='--', label='GQ=30')
    axes[1][2].set_title('Genotype Quality (GQ) at PASS sites')
    axes[1][2].set_xlabel('GQ score')
    axes[1][2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Custom filtering demonstration ---
# Apply stricter filters beyond VQSR/hard filtering for a clean analysis set

if not df.empty:
    raw_pass = df[df['filter'] == 'PASS']

    high_quality = raw_pass[
        (raw_pass['gq'] >= 30) &     # High genotype confidence
        (raw_pass['dp'] >= 10) &     # Minimum depth
        (raw_pass['af'].notna()) &   # Allele frequency calculable
        (raw_pass['af'] >= 0.2) &    # Remove very low AF (likely artifacts)
        (raw_pass['qd'].notna()) &
        (raw_pass['qd'] >= 5)
    ]

    print(f"Raw PASS variants:          {len(raw_pass):>8,}")
    print(f"After GQ>=30 + DP>=10:      {len(high_quality):>8,} ({100*len(high_quality)/len(raw_pass):.1f}% retained)")
    print(f"\nHigh-quality SNPs: {len(high_quality[high_quality['vtype']=='SNP']):,}")
    print(f"High-quality indels: {len(high_quality[high_quality['vtype'].isin(['INS','DEL'])]):,}")